In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from pathlib import Path

XLSX = Path('/content/drive/MyDrive/results.xlsx')
print('exists:', XLSX.exists())
xl = pd.ExcelFile(XLSX)
for s in xl.sheet_names:
    d_ = xl.parse(s)
    print(f'\n=== {s}  {d_.shape}')
    print(d_.head(12).to_string(index=False))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

DRIVE = Path('/content/drive/MyDrive')
FIG = DRIVE / 'diss-figures'
FIG.mkdir(parents=True, exist_ok=True)
XLSX = DRIVE / 'results.xlsx'
MASTER = DRIVE / 'chbmit_master.csv'

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
    'grid.alpha': 0.3,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'figure.facecolor': 'white',
})

OK = {'blue': '#0072B2', 'orange': '#E69F00', 'green': '#009E73',
      'red': '#D55E00', 'purple': '#CC79A7', 'sky': '#56B4E9',
      'yellow': '#F0E442', 'grey': '#666666'}

TODO = []
def flag(msg):
    TODO.append(msg)

def save(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(FIG / f'{name}.{ext}')
    print(f'saved {name}')

In [ ]:
DATASETS = ['MP50', 'CHB-MIT', 'MP8', 'Rowing', 'CMJ', 'SYNTH']
xl = pd.ExcelFile(XLSX)

def sheet(name, h=2):
    return xl.parse(name, header=h)

A = sheet('Accuracy').dropna(subset=['Model'])
A = A[A['Model'].isin(['MantisV2', 'Chronos-2', 'TiViT', 'MOMENT'])].set_index('Model')
Av = A[DATASETS].apply(pd.to_numeric, errors='coerce')
ACC = {('TiViT (DINOv2)' if m == 'TiViT' else m): Av.loc[m].tolist() for m in Av.index}
TB = sheet('TiViT backbone comparison').set_index('Dataset')
ACC['TiViT (DINOv2)'][DATASETS.index('CHB-MIT')] = float(pd.to_numeric(TB.loc['CHB-MIT', 'TiViT-DINOv2']))

R = sheet('Rocket')
R = R[R['Method'] == 'MiniRocket']
mv = R[R['Variant'] == 'multivariate'].set_index('Dataset')['Accuracy'].astype(float)
sc = R[R['Variant'] == 'single-channel'].set_index('Dataset')['Accuracy'].astype(float)
BASELINE_NAME = 'MiniRocket'
BASELINE = [float(mv.get(d, np.nan)) for d in DATASETS]
GAP = {d: float(mv[d] - sc[d]) for d in DATASETS if d in mv.index and d in sc.index}
missing_gap = [d for d in DATASETS if d not in GAP]
if missing_gap:
    flag(f'ROCKET single-channel rows missing for {missing_gap}')

def abl(sheet_name):
    t = sheet(sheet_name).dropna(subset=['Ablation'])
    t = t[~t['Ablation'].astype(str).str.startswith('Note')]
    out = {}
    for _, r in t.iterrows():
        c = str(r['Ablation'])
        if c in out:
            continue
        out[c] = {5: float(r['Sens @5% FPR']),
                  10: float(r['Sens @10% FPR']),
                  20: float(r['Sens @20% FPR'])}
    return out

ABL = {'Chronos-2 (frozen probe)': abl('RQ1 event-level ablations'),
       'MOMENT (fine-tuned head)': abl('RQ1 MOMENT ablations')}

E = sheet('RQ3 event sensitivity')
E['num'] = pd.to_numeric(E['Sens @5% FPR'], errors='coerce')
E = E.dropna(subset=['num'])
E = E[~E['Method'].astype(str).str.startswith(('chb', 'Patient'))]

SENS, HANDLING = {}, {}
for _, row in E.iterrows():
    m = str(row['Method'])
    if m in SENS:
        continue
    SENS[m] = {5: float(row['Sens @5% FPR']),
               10: float(row['Sens @10% FPR']),
               20: float(row['Sens @20% FPR'])}
    HANDLING[m] = str(row['Channel handling'])

print('RQ3 methods:', list(SENS.keys()))

Q4 = sheet('RQ4 retrofitting').dropna(subset=['Method'])
Q4 = Q4[~Q4.index.duplicated()] if Q4.index.has_duplicates else Q4
Q4 = Q4.drop_duplicates(subset=['Method']).set_index('Method')
RETROFIT = {}
for m in ['Chronos-2', 'MantisV2']:
    RETROFIT[m] = {
        'acc': (float(Q4.loc[f'{m} (baseline)', 'Accuracy']),
                float(Q4.loc[f'{m} + band+corr', 'Accuracy'])),
        'sens': {p: (float(Q4.loc[f'{m} (baseline)', f'Sens @{p}% FPR']),
                     float(Q4.loc[f'{m} + band+corr', f'Sens @{p}% FPR']))
                 for p in (5, 10, 20)}}

P = sheet('SigLIP2 per-channel', h=0).drop_duplicates(subset=['dataset']).set_index('dataset')
S = sheet('SigLIP2 stacked', h=0).drop_duplicates(subset=['dataset']).set_index('dataset')
Fu = sheet('SigLIP2 fused', h=0).drop_duplicates(subset=['dataset']).set_index('dataset')
sig_map = dict(zip(['MP50', 'CHBMIT', 'MP8', 'Rowing', 'CMJ', 'SYNTH'], DATASETS))
SIG = pd.DataFrame({
    'per-channel': P['ridge_acc'], 'stacked': S['stacked_acc'], 'fused': Fu['fused_acc']
}).rename(index=sig_map).loc[DATASETS]

PROBE = None
if MASTER.exists():
    M = pd.read_csv(MASTER)
    D_ = M[M.probe == 'default'].drop_duplicates(subset=['cond']).set_index('cond')
    T_ = M[M.probe == 'tuned'].drop_duplicates(subset=['cond']).set_index('cond')
    PROBE = {m: {'default': float(D_.loc[m, 'acc']),
                 'tuned': float(T_.loc[m, 'acc']),
                 'dim': int(D_.loc[m, 'dim'])}
             for m in ('Chronos-2', 'MantisV2') if m in D_.index and m in T_.index}
else:
    flag('chbmit_master.csv missing, probe confound figure will skip')

HEADROOM = None
flag('headroom values (2.3 / 14.7 / fused) not in workbook, figure skips until added')

ALPHA_CSV = DRIVE / 'alpha_sensitivity.csv'
if not ALPHA_CSV.exists():
    flag('alpha_sensitivity.csv missing, alpha figure will skip')

acc_chb = ACC['Chronos-2'][DATASETS.index('CHB-MIT')]
q4_chb = RETROFIT['Chronos-2']['acc'][0]
if abs(acc_chb - q4_chb) > 0.001:
    flag(f'Chronos-2 CHB-MIT differs between Accuracy sheet ({acc_chb}) and RQ4 sheet ({q4_chb}); decide which run is canonical')

print(f'{len(TODO)} flags')
for t in TODO:
    print(' -', t)

In [ ]:
lbl = [d for d in DATASETS if d in GAP]
val = [GAP[d] * 100 for d in lbl]

fig, ax = plt.subplots(figsize=(6.2, 2.8))
ax.bar(lbl, val, width=0.6, edgecolor='white', linewidth=0.4,
       color=[OK['blue'] if v > 0 else OK['red'] for v in val])
ax.axhline(0, lw=0.9, color='k')
for i, v in enumerate(val):
    ax.text(i, v + (0.15 if v > 0 else -0.15), f'{v:+.1f}',
            ha='center', va='bottom' if v > 0 else 'top', fontsize=7.5)
ax.set_ylabel('multivariate minus single-channel\nMiniRocket accuracy (pts)')
ax.yaxis.grid(True)
ax.set_axisbelow(True)
save(fig, 'fig5_0_rocket_gap')
plt.show()

In [ ]:
models = list(ACC.keys())
n, k = len(models) + 1, len(DATASETS)
x = np.arange(k)
w = 0.8 / n
cols = [OK['blue'], OK['orange'], OK['green'], OK['purple']]

fig, ax = plt.subplots(figsize=(7.2, 3.4))
for i, m in enumerate(models):
    pos = x + (i - n / 2 + 0.5) * w
    vals = np.array(ACC[m], dtype=float)
    ax.bar(pos, np.nan_to_num(vals), w, label=m, color=cols[i],
           edgecolor='white', linewidth=0.4)
    for xi, v in zip(pos, vals):
        if np.isnan(v):
            ax.text(xi, 0.412, 'not run', ha='center', va='bottom',
                    fontsize=6, color=OK['grey'], rotation=90)
ax.bar(x + (n - n / 2 - 0.5) * w, BASELINE, w, label=BASELINE_NAME,
       color='none', edgecolor=OK['grey'], hatch='///', linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels(DATASETS)
ax.set_ylim(0.4, 1.0)
ax.set_ylabel('window accuracy')
ax.yaxis.grid(True)
ax.set_axisbelow(True)
ax.annotate('', xy=(k - 0.45, -0.145), xytext=(-0.45, -0.145),
            xycoords=('data', 'axes fraction'),
            arrowprops=dict(arrowstyle='-|>', color=OK['grey'], lw=0.8))
ax.text(0.5, -0.20, 'decreasing measured cross-channel structure',
        transform=ax.transAxes, ha='center', va='top', fontsize=7.5, color=OK['grey'])
ax.legend(ncol=5, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.16))
save(fig, 'fig5_1_accuracy_matrix')
plt.show()

In [ ]:
short_map = {
    'Baseline (unmodified)': 'Intact',
    'Permuted channel order': 'Permuted',
    'Desynced channel timing': 'Desynced',
    'Channel-pooled (collapsed)': 'Pooled',
    'Channel-pooled (broadcast)': 'Pooled',
}
cc_map = {'Intact': OK['grey'], 'Permuted': OK['blue'],
          'Desynced': OK['orange'], 'Pooled': OK['red']}

fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.1), sharey=True)
for ax, (model, tbl) in zip(axes, ABL.items()):
    conds = list(tbl.keys())
    labels = [short_map.get(c, c) for c in conds]
    v5 = [tbl[c][5] for c in conds]
    colors = [cc_map.get(l, OK['purple']) for l in labels]
    ax.bar(labels, v5, color=colors, width=0.62, edgecolor='white', linewidth=0.4)
    for i, c in enumerate(conds):
        ax.plot([i - 0.1, i + 0.1], [tbl[c][10], tbl[c][20]], 'o',
                color='k', ms=3, alpha=0.55)
        ax.text(i, v5[i] + 0.010, f'{v5[i]:.3f}', ha='center', fontsize=7.5)
    base = tbl.get('Baseline (unmodified)', tbl[conds[0]])
    ax.axhline(base[5], ls='--', lw=0.9, color='k', zorder=0)
    ax.set_title(model)
    ax.set_ylim(0.72, 1.08)
    ax.tick_params(axis='x', rotation=15)
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
axes[0].set_ylabel('event sensitivity (bars: 5% FPR,\ndots: 10% and 20%)')
save(fig, 'fig5_2_ablation_events')
plt.show()

In [ ]:
order = ['MantisV2', 'SigLIP2 per-channel', 'Chronos-2', 'Classical band+corr',
         'SigLIP2 fused', 'SigLIP2 stacked']
order = [m for m in order if m in SENS]
val = [SENS[m][5] for m in order]
col = [OK['green'] if 'band' in m.lower()
       else OK['blue'] if 'stacked' in m or 'fused' in m or 'Chronos' in m
       else OK['orange'] for m in order]

fig, ax = plt.subplots(figsize=(6.4, 0.42 * len(order) + 1.2))
bars = ax.barh(order, val, color=col, height=0.62, edgecolor='white', linewidth=0.4)
ax.axvline(SENS['Chronos-2'][5], ls='--', lw=0.9, color=OK['grey'], zorder=0)
for b, v in zip(bars, val):
    ax.text(v + 0.004, b.get_y() + b.get_height() / 2, f'{v:.3f}',
            va='center', fontsize=8)
ax.set_xlim(0.72, 1.03)
ax.set_xlabel('event sensitivity at 5% interictal FPR (34 events, 4 unseen patients)')
ax.xaxis.grid(True)
ax.set_axisbelow(True)
save(fig, 'fig5_3_second_instrument')
plt.show()

In [ ]:
x = np.arange(len(DATASETS))
w = 0.26
arm_cols = {'per-channel': OK['orange'], 'stacked': OK['blue'], 'fused': OK['purple']}

fig, ax = plt.subplots(figsize=(7.2, 3.2))
for i, arm in enumerate(['per-channel', 'stacked', 'fused']):
    ax.bar(x + (i - 1) * w, SIG[arm].values, w, label=arm,
           color=arm_cols[arm], edgecolor='white', linewidth=0.4)
ax.set_xticks(x)
ax.set_xticklabels(DATASETS)
ax.set_ylim(0.3, 1.0)
ax.set_ylabel('window accuracy, frozen SigLIP2')
ax.yaxis.grid(True)
ax.set_axisbelow(True)
mp50 = DATASETS.index('MP50')
ax.annotate('stacked collapses\nwhen 50 channels\nshare one image',
            xy=(mp50, SIG.loc['MP50', 'stacked']),
            xytext=(mp50 + 0.9, 0.44), fontsize=7.5, color=OK['grey'],
            arrowprops=dict(arrowstyle='->', color=OK['grey'], lw=0.8))
ax.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.14))
save(fig, 'fig5_4_siglip2_arms')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
mcol = {'Chronos-2': OK['blue'], 'MantisV2': OK['orange']}

panels = [('acc', 'window accuracy'), ('sens', 'event sensitivity at 5% FPR')]
for ax, (key, lab) in zip(axes, panels):
    pairs = {m: (RETROFIT[m]['acc'] if key == 'acc' else RETROFIT[m]['sens'][5])
             for m in RETROFIT}
    allv = [v for p in pairs.values() for v in p]
    pad = (max(allv) - min(allv)) * 0.28
    for m, (a, b) in pairs.items():
        ax.annotate('', xy=(1, b), xytext=(0, a),
                    arrowprops=dict(arrowstyle='-|>', color=mcol[m], lw=1.6,
                                    shrinkA=0, shrinkB=0))
        ax.plot([0, 1], [a, b], 'o', color=mcol[m], ms=5)
        ax.text(-0.06, a, f'{a:.3f}', ha='right', va='center', fontsize=8, color=mcol[m])
        ax.text(1.06, b, f'{b:.3f}', ha='left', va='center', fontsize=8, color=mcol[m])
    ax.set_xlim(-0.6, 1.6)
    ax.set_ylim(min(allv) - pad, max(allv) + pad)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['baseline', '+ band+corr'])
    ax.set_title(lab)
    ax.spines['bottom'].set_visible(False)
    ax.tick_params(axis='x', length=0)
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)

handles = [plt.Line2D([], [], color=c, lw=1.6, marker='o', ms=5, label=m)
           for m, c in mcol.items()]
axes[0].legend(handles=handles, frameon=False, loc='upper left')
save(fig, 'fig5_5_retrofit_dissociation')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.0))
xs = np.arange(3)
w = 0.34
for i, m in enumerate(RETROFIT):
    delta = [(RETROFIT[m]['sens'][p][1] - RETROFIT[m]['sens'][p][0]) * 100
             for p in (5, 10, 20)]
    ax.bar(xs + (i - 0.5) * w, delta, w, label=m, color=mcol[m],
           edgecolor='white', linewidth=0.4)
    for xi, d in zip(xs + (i - 0.5) * w, delta):
        ax.text(xi, d + (0.15 if d >= 0 else -0.15), f'{d:+.1f}',
                ha='center', va='bottom' if d >= 0 else 'top', fontsize=7.5)
ax.axhline(0, lw=0.9, color='k')
ax.set_xticks(xs)
ax.set_xticklabels(['5% FPR', '10% FPR', '20% FPR'])
ax.set_ylabel('change in event sensitivity\nafter retrofit (pts)')
ax.legend(frameon=False)
ax.yaxis.grid(True)
ax.set_axisbelow(True)
save(fig, 'fig5_6_retrofit_thresholds')
plt.show()

In [ ]:
if PROBE is None:
    print('skipped: run the master cell first, then re-run Cell 2')
else:
    models = list(PROBE.keys())
    x = np.arange(len(models))
    w = 0.34
    dflt = [PROBE[m]['default'] for m in models]
    tund = [PROBE[m]['tuned'] for m in models]

    fig, ax = plt.subplots(figsize=(5.8, 3.2))
    ax.bar(x - w / 2, dflt, w, label='library default', color=OK['grey'],
           edgecolor='white', linewidth=0.4)
    ax.bar(x + w / 2, tund, w, label='tuned', color=OK['blue'],
           edgecolor='white', linewidth=0.4)
    for xi, (a, b) in enumerate(zip(dflt, tund)):
        ax.text(xi - w / 2, a + 0.004, f'{a:.3f}', ha='center', fontsize=8)
        ax.text(xi + w / 2, b + 0.004, f'{b:.3f}', ha='center', fontsize=8)
    for off, vals, c in [(-w / 2, dflt, OK['grey']), (w / 2, tund, OK['blue'])]:
        lo, hi = sorted(vals)
        ax.annotate('', xy=(1 + off, lo), xytext=(1 + off, hi),
                    arrowprops=dict(arrowstyle='<->', color=c, lw=0.9))
        ax.text(1 + off + 0.05, (lo + hi) / 2, f'{(hi - lo) * 100:.1f} pts',
                va='center', fontsize=7.5, color=c)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{m}\n({PROBE[m]['dim']} dim)" for m in models])
    ax.set_ylim(min(dflt) - 0.06, max(tund) + 0.05)
    ax.set_ylabel('window accuracy, CHB-MIT')
    ax.legend(frameon=False, loc='upper left')
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
    save(fig, 'fig5_7_probe_confound')
    plt.show()

In [ ]:
ALPHA_CSV = DRIVE / 'alpha_sensitivity.csv'
if not ALPHA_CSV.exists():
    print('skipped: save the alpha results frame to alpha_sensitivity.csv')
else:
    a = pd.read_csv(ALPHA_CSV)
    keys = sorted(a.model.unique())
    pal = [OK['blue'], OK['orange'], OK['green'], OK['red'], OK['purple'], OK['grey']]
    cmap = dict(zip(keys, pal))

    fig, ax = plt.subplots(1, 2, figsize=(7.6, 3.2))
    for m in keys:
        s = a[a.model == m]
        ax[0].scatter(s.dim, s.alpha, s=34, alpha=.9, color=cmap[m], label=m)
        ax[1].scatter(s.dim, s.delta * 100, s=34, alpha=.9, color=cmap[m], label=m)
    ax[0].axhline(1.0, ls='--', c='k', lw=0.9)
    ax[0].text(a.dim.min(), 1.35, 'library default', fontsize=7.5)
    ax[0].set(xscale='log', yscale='log', xlabel='embedding dimension',
              ylabel='selected $\\alpha$')
    ax[0].set_title('Selected regularisation rises with width')
    r = np.corrcoef(np.log10(a.dim), a.delta)[0, 1]
    ax[1].axhline(0, ls='--', c='k', lw=0.9)
    ax[1].set(xscale='log', xlabel='embedding dimension',
              ylabel='accuracy gain from tuning (pts)')
    ax[1].set_title('Gains vary by an order of magnitude')
    ax[1].text(0.97, 0.05, f'r = {r:.2f} (log dim)', transform=ax[1].transAxes,
               ha='right', fontsize=7.5, color=OK['grey'])
    ax[1].legend(fontsize=7, frameon=False, loc='upper left')
    for a_ in ax:
        a_.yaxis.grid(True)
        a_.set_axisbelow(True)
    save(fig, 'fig5_8_alpha_sensitivity')
    plt.show()

In [ ]:
print('FILES ON DRIVE')
for f in sorted(FIG.glob('fig5_*.pdf')):
    print(' ', f.name, f'{f.stat().st_size / 1024:.0f} KB')
print()
print(f'OUTSTANDING ({len(TODO)})')
for t in TODO:
    print(' -', t)

In [ ]:
from pathlib import Path
import numpy as np

DRIVE = Path('/content/drive/MyDrive')
DD = DRIVE / 'diss-datasets'

hits = sorted(DD.rglob('*CHBMIT*')) + sorted(DD.rglob('*chbmit*'))
for h in hits:
    print(h.relative_to(DRIVE), f'{h.stat().st_size/1e6:.0f} MB' if h.is_file() else 'dir')

emb = sorted(DD.rglob('*CHBMIT*emb.npy'))
print()
print('embedding files:', [e.name for e in emb])
need = ['Chronos2_CHBMIT_v2_train_emb.npy', 'Chronos2_CHBMIT_v2_test_emb.npy',
        'MantisV2_CHBMIT_v2_train_emb.npy', 'MantisV2_CHBMIT_v2_test_emb.npy']
missing = [n for n in need if not any(e.name == n for e in emb)]
print('MISSING:' if missing else 'all four embeddings present', missing if missing else '')

In [ ]:
TEST_PATS = {'chb20', 'chb22', 'chb23', 'chb24'}
PARTS = sorted((DD / 'chbmit_parts_v2').glob('chb*.npy'))

tr, te = {'X': [], 'y': [], 'file': [], 't0': [], 'patient': []}, \
         {'X': [], 'y': [], 'file': [], 't0': [], 'patient': []}

for p in PARTS:
    obj = np.load(p, allow_pickle=True).item()
    dst = te if p.stem in TEST_PATS else tr
    for k in dst:
        dst[k].append(obj[k])

tr = {k: np.concatenate(v) for k, v in tr.items()}
te = {k: np.concatenate(v) for k, v in te.items()}

rng = np.random.default_rng(1312)
perm = rng.permutation(len(tr['y']))
for k in tr:
    tr[k] = tr[k][perm]

d = {'train': tr, 'test': te}
labels = {'CHBMIT': (tr['y'], te['y'])}
SAVE_DIR = DD

print('train:', tr['X'].shape, ' test:', te['X'].shape)
print('first 10 train labels:', tr['y'][:10])

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import RidgeClassifier
print('cv:', cross_val_score(RidgeClassifier(alpha=100),
      np.load(DD/'Chronos2_CHBMIT_v2_train_emb.npy'), tr['y'], cv=3).mean())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeClassifier, RidgeClassifierCV
from sklearn.metrics import accuracy_score, f1_score

ALPHAS = np.logspace(-3, 3, 13)
FS = 256
BANDS = [(0.5, 4), (4, 8), (8, 13), (13, 30), (30, 64)]

ytr, yte = labels['CHBMIT']
Xtr, Xte = d['train']['X'], d['test']['X']
file_te = np.asarray(d['test']['file'])
t0_te = d['test']['t0'].astype(int)
pat_te = np.asarray(d['test']['patient'])

ev = np.full(len(yte), -1, int)
nx = 0
for f in np.unique(file_te):
    ix = np.where((file_te == f) & (yte == 1))[0]
    if not len(ix):
        continue
    o = ix[np.argsort(t0_te[ix])]
    s = t0_te[o]
    loc = np.zeros(len(o), int)
    for i in range(1, len(o)):
        loc[i] = loc[i - 1] + (1 if s[i] - s[i - 1] > 30 else 0)
    for l, g in zip(loc, o):
        ev[g] = nx + l
    nx += int(loc.max()) + 1
neg = yte == 0
NEV = nx
print(f'{NEV} events')

def sens(pr, tgt=0.05):
    best = 0.0
    for t in np.unique(pr):
        p = pr >= t
        if p[neg].mean() > tgt:
            continue
        best = max(best, np.mean([p[ev == e].any() for e in range(NEV)]))
    return best

def min_fpr_full_coverage(pr):
    best = 1.0
    for t in np.unique(pr):
        p = pr >= t
        if all(p[ev == e].any() for e in range(NEV)):
            best = min(best, p[neg].mean())
    return best

def band_power(X):
    fr = np.fft.rfftfreq(X.shape[2], 1 / FS)
    ps = np.abs(np.fft.rfft(X, axis=2)) ** 2
    return np.concatenate([np.log(ps[:, :, (fr >= lo) & (fr < hi)].sum(2) + 1e-8)
                           for lo, hi in BANDS], 1)

def corr_f(X):
    iu = np.triu_indices(X.shape[1], 1)
    return np.nan_to_num(np.stack([np.corrcoef(X[i])[iu] for i in range(len(X))]))

BP_tr, BP_te = band_power(Xtr), band_power(Xte)
CR_tr, CR_te = corr_f(Xtr), corr_f(Xte)
CL_tr = np.concatenate([BP_tr, CR_tr], 1)
CL_te = np.concatenate([BP_te, CR_te], 1)

Zc_tr = np.load(SAVE_DIR / 'Chronos2_CHBMIT_v2_train_emb.npy')
Zc_te = np.load(SAVE_DIR / 'Chronos2_CHBMIT_v2_test_emb.npy')
Zm_tr = np.load(SAVE_DIR / 'MantisV2_CHBMIT_v2_train_emb.npy')
Zm_te = np.load(SAVE_DIR / 'MantisV2_CHBMIT_v2_test_emb.npy')

CONDS = {
    'Chronos-2': (Zc_tr, Zc_te),
    'MantisV2': (Zm_tr, Zm_te),
    'band power only': (BP_tr, BP_te),
    'correlation only': (CR_tr, CR_te),
    'band+corr': (CL_tr, CL_te),
    'Chronos-2 + classical': (np.concatenate([Zc_tr, CL_tr], 1), np.concatenate([Zc_te, CL_te], 1)),
    'MantisV2 + classical': (np.concatenate([Zm_tr, CL_tr], 1), np.concatenate([Zm_te, CL_te], 1)),
}

out = []
for tag, (a, b) in CONDS.items():
    for lab, clf in [('default', RidgeClassifier()),
                     ('tuned', RidgeClassifierCV(alphas=ALPHAS))]:
        c = clf.fit(a, ytr)
        p = c.predict(b)
        scores = c.decision_function(b)
        row = dict(cond=tag, probe=lab, dim=a.shape[1],
                   alpha=getattr(c, 'alpha_', 1.0),
                   acc=accuracy_score(yte, p),
                   f1=f1_score(yte, p, average='macro'),
                   sens5=sens(scores),
                   sens10=sens(scores, 0.10),
                   sens20=sens(scores, 0.20),
                   min_fpr_full=min_fpr_full_coverage(scores))
        out.append(row)
        print(f"{tag:24} {lab:8} acc {row['acc']:.4f}  sens@5% {row['sens5']:.3f}  "
              f"alpha {row['alpha']:g}  minFPR {row['min_fpr_full']:.3f}")

pd.DataFrame(out).to_csv('/content/drive/MyDrive/chbmit_master.csv', index=False)
print('\nsaved chbmit_master.csv')

In [ ]:
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import cross_val_score

ftr = d['train']['file']
ttr = d['train']['t0']
ptr = d['train']['patient']

orders = {
    'as-loaded (patient files sorted)': np.arange(len(ytr)),
    'global sort by (file, t0)': np.lexsort((ttr, ftr)),
    'sort by (patient, t0)': np.lexsort((ttr, ptr)),
}
for name, o in orders.items():
    cv = cross_val_score(RidgeClassifier(alpha=100), Zc_tr, ytr[o], cv=3).mean()
    print(f'{name:36} cv {cv:.3f}')

In [ ]:
import json, re
nb = json.load(open('/content/drive/MyDrive/Colab Notebooks/Untitled8.ipynb'))
for i, c in enumerate(nb['cells']):
    if c['cell_type'] != 'code':
        continue
    src = ''.join(c['source'])
    if re.search(r"chbmit_parts|d\s*=\s*{|'train'.*'X'", src):
        print(f'===== cell {i} =====')
        print(src[:1500])
        print()

In [ ]:
import json, re
nb = json.load(open('/content/drive/MyDrive/Colab Notebooks/Untitled8.ipynb'))
for i, c in enumerate(nb['cells']):
    if c['cell_type'] != 'code':
        continue
    src = ''.join(c['source'])
    if re.search(r"chb\d\d|chbmit|CHBMIT", src) and 'labels[' not in src.split('\n')[0]:
        print(f'===== cell {i} ({len(src)} chars) =====')
        print(src[:2000])
        print()

In [ ]:
import numpy as np

SK = DD / 'siglip2_kaggle'
ARMS = {
    'per-channel': SK / 'SigLIP2_CHBMIT_test_scores.npz',
    'stacked':     SK / 'SigLIP2stack_CHBMIT_test_scores.npz',
    'fused':       SK / 'SigLIP2fused_CHBMIT_test_scores.npz',
}

head, det5 = {}, {}
for arm, path in ARMS.items():
    z = np.load(path, allow_pickle=True)
    key = [k for k in z.keys() if 'score' in k.lower() or 'decision' in k.lower()]
    scores = z[key[0]] if key else z[list(z.keys())[0]]
    print(arm, 'keys:', list(z.keys()), 'shape:', scores.shape)
    assert len(scores) == len(yte), f'{arm}: {len(scores)} vs {len(yte)} rows'
    head[arm] = min_fpr_full_coverage(scores)
    for t in np.unique(scores):
        p = scores >= t
        if p[neg].mean() <= 0.05:
            det5[arm] = np.array([p[ev == e].any() for e in range(NEV)])
            break

print()
for arm, v in head.items():
    print(f'{arm:12} min FPR for 34/34: {v*100:.1f}%')

pats_ev = np.array([pat_te[ev == e][0] for e in range(NEV)])
b = det5['stacked'] & ~det5['per-channel']
c = det5['per-channel'] & ~det5['stacked']
n_disc = int(b.sum() + c.sum())
from math import comb
k = int(min(b.sum(), c.sum()))
p_exact = sum(comb(n_disc, i) for i in range(0, k + 1)) / 2**n_disc * 2 if n_disc else 1.0
print(f'\nMcNemar: stacked-only {int(b.sum())}, per-channel-only {int(c.sum())}, '
      f'discordant {n_disc}, exact two-sided p = {min(p_exact, 1.0):.3f}')
print('discordant events by patient:', pats_ev[b | c])

In [ ]:
import numpy as np
from math import comb

SK = DD / 'siglip2_kaggle'
ARMS = {
    'per-channel': SK / 'SigLIP2_CHBMIT_test_scores.npz',
    'stacked':     SK / 'SigLIP2stack_CHBMIT_test_scores.npz',
    'fused':       SK / 'SigLIP2fused_CHBMIT_test_scores.npz',
}

def eval_arm(path):
    z = np.load(path, allow_pickle=True)
    fkey = 'rec_file' if 'rec_file' in z else 'source_file'
    y = z['y'].astype(int)
    t0 = z['t0'].astype(int)
    fl = np.asarray(z[fkey]).astype(str)
    pat = np.asarray(z['patient']).astype(str)
    sc = z['scores']

    evl = np.full(len(y), -1, int)
    keys = {}
    nx = 0
    for f in np.unique(fl):
        ix = np.where((fl == f) & (y == 1))[0]
        if not len(ix):
            continue
        o = ix[np.argsort(t0[ix])]
        s = t0[o]
        loc = np.zeros(len(o), int)
        for i in range(1, len(o)):
            loc[i] = loc[i - 1] + (1 if s[i] - s[i - 1] > 30 else 0)
        for l, g in zip(loc, o):
            evl[g] = nx + l
        for l in np.unique(loc):
            first_t0 = s[loc == l].min()
            keys[nx + l] = (f, int(first_t0))
        nx += int(loc.max()) + 1
    ng = y == 0

    n_ev = nx
    hr = 1.0
    det5 = None
    for t in np.unique(sc):
        p = sc >= t
        if all(p[evl == e].any() for e in range(n_ev)):
            hr = min(hr, p[ng].mean())
    for t in np.unique(sc):
        p = sc >= t
        if p[ng].mean() <= 0.05:
            det5 = {keys[e]: bool(p[evl == e].any()) for e in range(n_ev)}
            s5 = np.mean([p[evl == e].any() for e in range(n_ev)])
            break
    return dict(n_events=n_ev, headroom=hr, sens5=s5, det5=det5,
                pat_of={k: k[0][:5] for k in det5})

R = {arm: eval_arm(p) for arm, p in ARMS.items()}
for arm, r in R.items():
    print(f"{arm:12} events {r['n_events']}  sens@5% {r['sens5']:.3f}  "
          f"min FPR full coverage {r['headroom']*100:.1f}%")

common = set(R['stacked']['det5']) & set(R['per-channel']['det5'])
b = [k for k in common if R['stacked']['det5'][k] and not R['per-channel']['det5'][k]]
c = [k for k in common if R['per-channel']['det5'][k] and not R['stacked']['det5'][k]]
n = len(b) + len(c)
k_ = min(len(b), len(c))
p_exact = min(1.0, sum(comb(n, i) for i in range(0, k_ + 1)) / 2**n * 2) if n else 1.0
print(f"\nMcNemar stacked vs per-channel: +{len(b)} / -{len(c)}, "
      f"discordant {n}, exact two-sided p = {p_exact:.3f}")
print('stacked-only events:', [(f[:8], t) for f, t in b])
print('per-channel-only events:', [(f[:8], t) for f, t in c])

In [ ]:
def sens_at(path, tgt):
    z = np.load(path, allow_pickle=True)
    fkey = 'rec_file' if 'rec_file' in z else 'source_file'
    y = z['y'].astype(int); t0 = z['t0'].astype(int)
    fl = np.asarray(z[fkey]).astype(str); sc = z['scores']
    evl = np.full(len(y), -1, int); nx = 0
    for f in np.unique(fl):
        ix = np.where((fl == f) & (y == 1))[0]
        if not len(ix): continue
        o = ix[np.argsort(t0[ix])]; s = t0[o]
        loc = np.zeros(len(o), int)
        for i in range(1, len(o)):
            loc[i] = loc[i-1] + (1 if s[i]-s[i-1] > 30 else 0)
        for l, g in zip(loc, o): evl[g] = nx + l
        nx += int(loc.max()) + 1
    ng = y == 0
    best = 0.0
    for t in np.unique(sc):
        p = sc >= t
        if p[ng].mean() > tgt: continue
        best = max(best, np.mean([p[evl == e].any() for e in range(nx)]))
    return best

for arm, path in ARMS.items():
    print(f"{arm:12}", " ".join(f"@{int(t*100)}%: {sens_at(path, t):.3f}" for t in (0.05, 0.10, 0.20)))

print('fused chb20 detected:',
      sum(1 for k, v in R['fused']['det5'].items() if k[0].startswith('chb20') and v), 'of 8')